<a href="https://colab.research.google.com/github/Mankz111/machine-learning-playground/blob/main/portugal_housing_model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np
from sklearn.experimental import enable_iterative_imputer
from sklearn.impute import IterativeImputer
from sklearn.ensemble import RandomForestRegressor
import category_encoders as ce
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error
import matplotlib.pyplot as plt
from sklearn.linear_model import BayesianRidge
import joblib

In [ ]:
data = pd.read_csv("/content/portugal_dataset.csv", low_memory=False)

In [ ]:
cols_to_drop = ["BuiltArea", "LotSize", "NumberOfWC", "ElectricCarsCharging", "PublishDate", "Floor", "HasParking", "GrossArea", "TotalRooms", "EnergyEfficiencyLevel"]

In [ ]:
data.drop(columns=cols_to_drop, axis=1, inplace=True)

In [ ]:
data = data[(data["NumberOfBathrooms"] >= 0) & (data["NumberOfBathrooms"] < 10)]
def iqr_calc(series, multiplier=1.5, floor=None):
  q1 = series.quantile(0.25)
  q3 = series.quantile(0.75)
  iqr = q3 - q1
  lower_limit = q1 - multiplier * iqr
  upper_limit = q3 + multiplier * iqr

  if floor is not None:
    lower_limit = max(lower_limit, floor)

  return lower_limit, upper_limit

for col, mult, min_val in [('LivingArea', 3.0, 10), ('TotalArea', 3.0, 10)]:
  lower_limit, upper_limit = iqr_calc(data[col], multiplier=mult, floor = min_val)
  data = data[(data[col] >= lower_limit) & (data[col] <= upper_limit)]

data = data[(data["LivingArea"] <= data["TotalArea"])]

In [ ]:
energy_map = {
    'A+': 8, 'A': 7, 'B': 6, 'B-': 5, 'C': 4,
    'D': 3, 'E': 2, 'F': 1, 'G': 1, 'NC': 0, 'No Certificate': 0
}

In [ ]:
data['EnergyCertificate'] = data['EnergyCertificate'].map(energy_map).fillna(0).astype(int)

In [ ]:
conservation_map = {
    'New': 6,
    'Like new': 5,
    'Good condition': 4,
    'Used': 3,
    'Reasonable': 2,
    'Needs renovation': 1
}

In [ ]:
data['ConservationStatus'] = data['ConservationStatus'].map(conservation_map).fillna(0).astype(int)

In [ ]:
garage_map = {True: 1, False: 0}

In [ ]:
data['Garage'] = data['Garage'].map(garage_map)

In [ ]:
data['Garage'].unique()

array([nan,  0.,  1.])

In [ ]:
data['Garage'] = data['Garage'].fillna(0).round().clip(0, 1).astype(int)

In [ ]:
data = data.dropna(subset=['Price'])

In [ ]:
target_cols = ['District', 'City', 'Town', 'Type']

In [ ]:
limiar = 2000000
data = data[data['Price'] <= limiar].copy()
X = data.drop('Price', axis=1)
y = data['Price']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [ ]:
encoder = ce.TargetEncoder(cols=target_cols, smoothing=10)


In [ ]:
X_train_encoder = encoder.fit_transform(X_train, y_train)
X_test_encoder = encoder.transform(X_test)

In [ ]:
train_for_imputation = pd.concat([X_train_encoder], axis=1)

In [ ]:
full_cols_to_impute = train_for_imputation.columns.tolist()

In [ ]:
it_imputer = IterativeImputer(
    estimator=RandomForestRegressor(n_estimators=50, n_jobs=-1, random_state=42),
    max_iter=10,
    random_state=42,
    verbose=2
)

# Testar KNNimputer para mais rapidez e talvez eficácia

In [ ]:
train_imputed_array = it_imputer.fit_transform(train_for_imputation)

[IterativeImputer] Completing matrix with shape (61925, 14)
[IterativeImputer] Ending imputation round 1/10, elapsed time 206.96
[IterativeImputer] Change: 78.47596279437948, scaled tolerance: 1006.7192586097441 
[IterativeImputer] Early stopping criterion reached.


In [ ]:
train_final = pd.DataFrame(train_imputed_array, columns=train_for_imputation.columns, index=X_train.index)

In [ ]:
cols_to_fix = ['NumberOfBedrooms', 'NumberOfBathrooms', 'ConstructionYear', 'Parking']

In [ ]:
for cols in cols_to_fix:
  train_final[cols] = train_final[cols].round().astype(int)

In [ ]:
print(train_final.isna().sum())

District              0
City                  0
Town                  0
Type                  0
EnergyCertificate     0
TotalArea             0
Parking               0
ConstructionYear      0
Garage                0
Elevator              0
NumberOfBedrooms      0
ConservationStatus    0
LivingArea            0
NumberOfBathrooms     0
dtype: int64


In [ ]:
data['ConstructionYear'].unique()

array([  nan, 1990., 2009., 2003., 1992., 1985., 1950., 1986., 2022.,
       1958., 1987., 1976., 2004., 1978., 1975., 1937., 1982., 1994.,
       1999., 1984., 2023., 1980., 2008., 1974., 2010., 2015., 1993.,
       1983., 2001., 1951., 1988., 1989., 2002., 2006., 1997., 1991.,
       1981., 1972., 2016., 1979., 2012., 2005., 2021., 2007., 1966.,
       2011., 2020., 1998., 1970., 2000., 1977., 1940., 1996., 1995.,
       2019., 2014., 2013., 1962., 1968., 1961., 1952., 1969., 1973.,
       1963., 1949., 1957., 1947., 1953., 1964., 1938., 1942., 1946.,
       1965., 1920., 1934., 2017., 1971., 1941., 1944., 1930., 1943.,
       1960., 1935., 1959., 1939., 2018., 1956., 1967., 1945., 1955.,
       1948., 1932., 1931., 1923., 1927., 1924., 1910., 1912., 1933.,
       1918., 1936., 1904., 1954., 1925., 1921., 1908., 1905., 1929.,
       1922., 1911., 1928., 1926., 1916., 1906., 1902., 1913., 2024.,
       1919., 1900., 1909., 1903., 1915., 2025.])

In [ ]:
test_for_imputation = pd.concat([X_test_encoder], axis=1)
test_imputed_array = it_imputer.transform(test_for_imputation)
test_final = pd.DataFrame(test_imputed_array, columns=test_for_imputation.columns, index=X_test.index)


[IterativeImputer] Completing matrix with shape (15482, 14)
[IterativeImputer] Ending imputation round 1/1, elapsed time 0.21


In [ ]:
test_final_features = pd.DataFrame(
    test_imputed_array,
    columns=X_train_encoder.columns,
    index=X_test.index
)
for col in cols_to_fix:
    test_final_features[col] = test_final_features[col].round().astype(int)

test_final = pd.concat([test_final_features, y_test], axis=1)

In [ ]:
import numpy as np
import lightgbm as lgb
from sklearn.metrics import mean_absolute_error, r2_score
import optuna

# --- 1. PREPARAÇÃO DA ÂNCORA DE VALOR ---
train_final['Price'] = y_train
test_final['Price'] = y_test

# Calculamos a âncora apenas com base no treino para evitar data leakage
train_final['m2_price'] = train_final['Price'] / (train_final['TotalArea'] + 1)
city_m2_map = train_final.groupby('City')['m2_price'].median()

# --- 2. ENGENHARIA DE ATRIBUTOS (APLICADA A AMBOS) ---
curr_year = 2025

for df in [train_final, test_final]:
    # Aplicar mapeamento da âncora
    df['City_Value_Anchor'] = df['City'].map(city_m2_map)

    # Preencher cidades que não existiam no treino com a mediana global
    df['City_Value_Anchor'] = df['City_Value_Anchor'].fillna(city_m2_map.median())

    # Criar variáveis de Idade e Interação
    df['Age'] = (curr_year - df['ConstructionYear']).clip(lower=0)
    df['Age_Squared'] = df['Age'] ** 2 # Para captar a curva em U (valor histórico)
    df['AgeLocationFactor'] = df['Age'] * df['City_Value_Anchor']


X_train_ready = train_final.drop(['Price', 'm2_price'], axis=1)
y_train_ready = train_final['Price']

X_test_ready = test_final.drop(['Price'], axis=1)
y_test_ready = test_final['Price']


def objective(trial):
    params = {
        'objective': 'huber',
        'metric': 'mae',
        'alpha': 0.85,
        'verbosity': -1,
        'boosting_type': 'gbdt',
        'random_state': 42,
        'learning_rate': trial.suggest_float('learning_rate', 0.005, 0.05),
        'num_leaves': trial.suggest_int('num_leaves', 100, 300),
        'feature_fraction': trial.suggest_float('feature_fraction', 0.6, 0.9),
        'bagging_fraction': trial.suggest_float('bagging_fraction', 0.6, 0.9),
        'min_data_in_leaf': trial.suggest_int('min_data_in_leaf', 10, 50),
        'lambda_l1': trial.suggest_float('lambda_l1', 0.01, 1.0),
        'lambda_l2': trial.suggest_float('lambda_l2', 0.01, 1.0)
    }

    y_train_log = np.log1p(y_train_ready)
    train_data = lgb.Dataset(X_train_ready, label=y_train_log)

    model = lgb.train(params, train_data, num_boost_round=2500)

    y_pred_log = model.predict(X_test_ready)
    y_pred_euros = np.expm1(y_pred_log)

    return mean_absolute_error(y_test_ready, y_pred_euros)

study = optuna.create_study(direction='minimize')
study.optimize(objective, n_trials=50)  # Ajuste o número de trials conforme necessário

best_params = study.best_params
print("Best hyperparameters:", best_params)

# Treinar o modelo final com os melhores hiperparâmetros
params = {
    'objective': 'huber',
    'metric': 'mae',
    'alpha': 0.85,
    'verbosity': -1,
    'boosting_type': 'gbdt',
    'random_state': 42,
    **best_params
}

y_train_log = np.log1p(y_train_ready)
train_data = lgb.Dataset(X_train_ready, label=y_train_log)

model = lgb.train(params, train_data, num_boost_round=2500)

y_pred_log = model.predict(X_test_ready)
y_pred_euros = np.expm1(y_pred_log)

mae = mean_absolute_error(y_test_ready, y_pred_euros)
r2 = r2_score(y_test_ready, y_pred_euros)

print(f"--- Resultados Finais ---")
print(f"MAE: {mae:.2f} €")
print(f"R²: {r2:.4f}")

[I 2025-12-29 20:49:54,221] A new study created in memory with name: no-name-e416b011-a0f3-43cc-a4ed-244e48059c84
[I 2025-12-29 20:51:19,783] Trial 0 finished with value: 64888.61724465755 and parameters: {'learning_rate': 0.005768089366297488, 'num_leaves': 126, 'feature_fraction': 0.6471157071842731, 'bagging_fraction': 0.7665426761795914, 'min_data_in_leaf': 50, 'lambda_l1': 0.31279635234825437, 'lambda_l2': 0.12931651061820257}. Best is trial 0 with value: 64888.61724465755.
[I 2025-12-29 20:51:49,552] Trial 1 finished with value: 56604.097269418955 and parameters: {'learning_rate': 0.021512410631998453, 'num_leaves': 135, 'feature_fraction': 0.6504728254966691, 'bagging_fraction': 0.8110727178790267, 'min_data_in_leaf': 28, 'lambda_l1': 0.1224351916659705, 'lambda_l2': 0.588560512683857}. Best is trial 1 with value: 56604.097269418955.
[I 2025-12-29 20:52:30,508] Trial 2 finished with value: 51642.6873491125 and parameters: {'learning_rate': 0.04516607077950322, 'num_leaves': 214,

Best hyperparameters: {'learning_rate': 0.03957564346851641, 'num_leaves': 286, 'feature_fraction': 0.7545550293065713, 'bagging_fraction': 0.7744982060172603, 'min_data_in_leaf': 12, 'lambda_l1': 0.04771498646301178, 'lambda_l2': 0.4803285704071205}
--- Resultados Finais ---
MAE: 48701.65 €
R²: 0.8726


In [ ]:
data.describe()

In [ ]:
model.save_model('lgbm_model.txt')

In [ ]:
joblib.dump(city_m2_map, 'city_m2_map.pkl')